# shopextract -- E-Commerce Product Intelligence

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umerkhan95/shopextract/blob/main/notebooks/shopextract_demo.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/umerkhan95/shopextract/blob/main/notebooks/shopextract_demo.ipynb)

**Extract, analyze, validate, and monitor product data from any online store.**

`shopextract` is a Python library that turns any e-commerce URL into structured product data. It automatically detects the underlying platform, discovers product pages, and extracts inventory using a tiered strategy -- preferring fast API calls, falling back to intelligent crawling when needed.

### What you can do with it

| Capability | Description |
|---|---|
| **Platform Detection** | Identify Shopify, WooCommerce, Magento, BigCommerce, Shopware, or custom stores |
| **Product Extraction** | Pull structured product data (title, price, images, GTIN, variants) from any store |
| **Catalog Analysis** | Price distributions, brand breakdowns, completeness scoring |
| **Product Matching** | Fuzzy title matching and exact GTIN lookup across catalogs |
| **Marketplace Validation** | Check readiness for Google Shopping, idealo, Amazon, eBay |
| **Price Monitoring** | Track price changes, new arrivals, and discontinued products over time |
| **Multi-Format Export** | CSV, JSON, Google Shopping XML, Pandas DataFrame, Parquet |

This notebook walks through every major feature with live examples.

> **Environment note:** This notebook is designed to work on Google Colab, Kaggle, and local Jupyter. Sections that require network or browser access (detection, extraction) are wrapped in `try/except` with sample data fallbacks. All analysis, validation, export, and quality sections work everywhere.

In [ ]:
# Install shopextract (skip if already installed locally)
!pip install shopextract -q

import shopextract

print(f"shopextract v{shopextract.__version__}")
print(f"Platforms supported: {', '.join(p.value for p in shopextract.Platform)}")
print(f"Extraction tiers:   {', '.join(t.value for t in shopextract.ExtractionTier)}")

---

## 1. Platform Detection

`detect()` probes a URL using HTTP headers, API endpoints, and HTML content analysis to identify the e-commerce platform. It returns a `PlatformResult` with the detected platform, a confidence score (0.0 - 1.0), and the specific signals that triggered detection.

Detection runs in under 3 seconds for most stores -- no browser needed.

> **Note:** This cell requires network access. On Colab/Kaggle it may work if outbound HTTP is allowed, but browser-based fallbacks are unavailable.

In [ ]:
# Detect a Shopify store
try:
    result = await shopextract.detect("https://brooklinen.com")
    print(f"Platform:   {result.platform.value}")
    print(f"Confidence: {result.confidence:.0%}")
    print(f"Signals:    {', '.join(result.signals)}")
except Exception as e:
    print(f"Platform detection not available in this environment: {e}")
    print("This feature requires network access to probe the target store.")

---

## 2. Product Extraction

`extract()` is the main workhorse. Given a store URL, it:

1. **Detects** the platform (or accepts a pre-detected one)
2. **Discovers** product URLs via APIs, sitemaps, or crawling
3. **Extracts** product data using a tiered fallback chain:
   - **Tier 1 -- API**: Platform-native JSON APIs (fastest, most reliable)
   - **Tier 2 -- UnifiedCrawl**: JSON-LD, OpenGraph, markdown parsing from a single page fetch
   - **Tier 3 -- CSS**: Browser-based extraction with CSS selectors
4. **Normalizes** everything into a unified `Product` dataclass

Each product includes: title, price, currency, description, images, SKU, GTIN, vendor, variants, and more.

> **Note:** Live extraction may require a browser runtime (crawl4ai) which is not available on Colab/Kaggle. If extraction fails, sample data is loaded so all remaining sections work.

In [ ]:
try:
    result = await shopextract.extract("https://brooklinen.com", max_urls=5)
    products = result.products
    from dataclasses import asdict
    products_as_dicts = [asdict(p) for p in products]
    print(f"Platform:        {result.platform.value}")
    print(f"Extraction Tier: {result.tier.value}")
    print(f"Quality Score:   {result.quality_score:.2f}")
    print(f"Products Found:  {result.product_count}")
    print(f"URLs Attempted:  {result.urls_attempted}")
    print(f"URLs Succeeded:  {result.urls_succeeded}")
except Exception as e:
    print(f"Live extraction not available in this environment: {e}")
    print("Using sample data for remaining demos...\n")
    products_as_dicts = [
        {
            "title": "Classic Percale Sheet Set - White",
            "price": 149.00,
            "currency": "USD",
            "vendor": "Brooklinen",
            "sku": "PERC-SHT-WHT-Q",
            "gtin": "0850031234567",
            "image_url": "https://www.brooklinen.com/cdn/shop/products/percale-sheets-white.jpg",
            "product_url": "https://www.brooklinen.com/products/classic-percale-sheet-set",
            "in_stock": True,
            "description": "Crisp, cool percale sheets in 100% long-staple cotton. 270 thread count.",
            "product_type": "Sheets",
            "condition": "new",
        },
        {
            "title": "Luxe Sateen Duvet Cover - Cream",
            "price": 199.00,
            "currency": "USD",
            "vendor": "Brooklinen",
            "sku": "SATN-DVT-CRM-Q",
            "gtin": "0850031234890",
            "image_url": "https://www.brooklinen.com/cdn/shop/products/sateen-duvet-cream.jpg",
            "product_url": "https://www.brooklinen.com/products/luxe-sateen-duvet-cover",
            "in_stock": True,
            "description": "Buttery smooth sateen weave with a subtle sheen. 480 thread count.",
            "product_type": "Bedding",
            "condition": "new",
        },
        {
            "title": "Down Alternative Comforter",
            "price": 249.00,
            "currency": "USD",
            "vendor": "Brooklinen",
            "sku": "DOWN-ALT-CMF-Q",
            "gtin": None,
            "image_url": "https://www.brooklinen.com/cdn/shop/products/comforter-down-alt.jpg",
            "product_url": "https://www.brooklinen.com/products/down-alternative-comforter",
            "in_stock": True,
            "description": "Hypoallergenic fill, all-season weight. Machine washable.",
            "product_type": "Bedding",
            "condition": "new",
        },
        {
            "title": "Luxe Sateen Pillowcase Set",
            "price": 59.00,
            "currency": "USD",
            "vendor": "Brooklinen",
            "sku": "SATN-PLC-WHT-S",
            "gtin": "0850031234222",
            "image_url": "https://www.brooklinen.com/cdn/shop/products/sateen-pillowcases.jpg",
            "product_url": "https://www.brooklinen.com/products/luxe-sateen-pillowcase-set",
            "in_stock": True,
            "description": "Set of 2 standard pillowcases in luxe sateen.",
            "product_type": "Bedding",
            "condition": "new",
        },
        {
            "title": "Waffle Bath Towel Set",
            "price": 79.00,
            "currency": "USD",
            "vendor": "Brooklinen",
            "sku": "WAFL-TWL-WHT-S",
            "gtin": None,
            "image_url": "https://www.brooklinen.com/cdn/shop/products/waffle-towel-set.jpg",
            "product_url": "https://www.brooklinen.com/products/waffle-bath-towel-set",
            "in_stock": True,
            "description": "Lightweight, quick-drying waffle weave towels.",
            "product_type": "Bath",
            "condition": "new",
        },
        {
            "title": "Super-Plush Bath Robe",
            "price": 98.00,
            "currency": "USD",
            "vendor": "Brooklinen",
            "sku": "ROBE-PLH-GRY-M",
            "gtin": "0850031234333",
            "image_url": "https://www.brooklinen.com/cdn/shop/products/plush-robe-grey.jpg",
            "product_url": "https://www.brooklinen.com/products/super-plush-bath-robe",
            "in_stock": False,
            "description": "Turkish cotton terry robe. Absorbent and cozy.",
            "product_type": "Bath",
            "condition": "new",
        },
        {
            "title": "Mulberry Silk Pillowcase",
            "price": 59.00,
            "currency": "USD",
            "vendor": "Brooklinen",
            "sku": "SILK-PLC-IVR-S",
            "gtin": "0850031234444",
            "image_url": "https://www.brooklinen.com/cdn/shop/products/silk-pillowcase.jpg",
            "product_url": "https://www.brooklinen.com/products/mulberry-silk-pillowcase",
            "in_stock": True,
            "description": "22-momme pure mulberry silk. Reduces friction on hair and skin.",
            "product_type": "Bedding",
            "condition": "new",
        },
        {
            "title": "Lightweight Quilt",
            "price": 299.00,
            "currency": "USD",
            "vendor": "Marialma",
            "sku": "QUILT-LW-NAV-Q",
            "gtin": None,
            "image_url": "https://www.brooklinen.com/cdn/shop/products/quilt-navy.jpg",
            "product_url": "https://www.brooklinen.com/products/lightweight-quilt",
            "in_stock": True,
            "description": "Garment-washed for a lived-in feel. Lightweight warmth.",
            "product_type": "Bedding",
            "condition": "new",
        },
        {
            "title": "Classic Percale Duvet Cover",
            "price": 169.00,
            "currency": "USD",
            "vendor": "Brooklinen",
            "sku": "PERC-DVT-WHT-Q",
            "gtin": "0850031234555",
            "image_url": "https://www.brooklinen.com/cdn/shop/products/percale-duvet-white.jpg",
            "product_url": "https://www.brooklinen.com/products/classic-percale-duvet-cover",
            "in_stock": True,
            "description": "Matte finish, crisp hand-feel. Gets softer with every wash.",
            "product_type": "Bedding",
            "condition": "new",
        },
        {
            "title": "Cashmere Throw Blanket",
            "price": 399.00,
            "currency": "USD",
            "vendor": "Marialma",
            "sku": "CSHM-THRW-CAM-1",
            "gtin": None,
            "image_url": "",
            "product_url": "https://www.brooklinen.com/products/cashmere-throw-blanket",
            "in_stock": False,
            "description": "",
            "product_type": "Throws",
            "condition": "new",
        },
    ]
    print(f"Loaded {len(products_as_dicts)} sample products")

In [ ]:
# Display products as a formatted table
import pandas as pd

df_display = pd.DataFrame([
    {
        "Title": p["title"][:50] + ("..." if len(p["title"]) > 50 else ""),
        "Price": f"${p['price']:.2f}" if isinstance(p["price"], (int, float)) else f"${p['price']}",
        "Vendor": p.get("vendor") or "--",
        "SKU": p.get("sku") or "--",
        "In Stock": p.get("in_stock", True),
    }
    for p in products_as_dicts
])

df_display

---

## 3. Catalog Analysis

Works everywhere -- no network required.

The analysis module computes aggregate statistics from product data -- pure computation. Feed it a list of product dicts and get back price ranges, brand distributions, completeness scores, and more.

In [ ]:
# Compute catalog statistics
stats = shopextract.analyze_products(products_as_dicts)

print(f"Total Products:     {stats.total_products}")
print(f"Price Range:        ${stats.price_range[0]:.2f} -- ${stats.price_range[1]:.2f}")
print(f"Average Price:      ${stats.avg_price:.2f}")
print(f"Median Price:       ${stats.median_price:.2f}")
print(f"In Stock:           {stats.in_stock}")
print(f"Out of Stock:       {stats.out_of_stock}")
print(f"Has GTIN:           {stats.has_gtin}")
print(f"Has Images:         {stats.has_images}")
print(f"Completeness Score: {stats.completeness_score:.1%}")

if stats.currencies:
    print(f"\nCurrencies: {dict(stats.currencies)}")

In [ ]:
# Price distribution across buckets
dist = shopextract.price_distribution(products_as_dicts)

print("Price Distribution")
print("=" * 35)
for bucket, count in dist.items():
    bar = "#" * count
    print(f"  ${bucket:>10s}  {count:>3d}  {bar}")

In [ ]:
# Brand breakdown -- percentage of catalog per brand/vendor
brands = shopextract.brand_breakdown(products_as_dicts)

if brands:
    print("Brand Share of Catalog")
    print("=" * 40)
    for brand, pct in list(brands.items())[:10]:
        bar = "#" * int(pct / 2)
        print(f"  {brand:<25s} {pct:5.1f}%  {bar}")
else:
    print("No brand/vendor data available in this catalog.")

In [ ]:
# Find price outliers
price_outliers = shopextract.outliers(products_as_dicts)

if price_outliers:
    print(f"Found {len(price_outliers)} price outlier(s):")
    for p in price_outliers:
        print(f"  {p['title']:<40s}  ${float(p['price']):>7.2f}")
else:
    print("No significant price outliers detected.")

---

## 4. Product Matching and Comparison

Works everywhere -- no network required.

When you have product data from multiple sources, `fuzzy_match()` finds the same product across catalogs using title similarity. `match_gtin()` does exact lookups by GTIN, EAN, UPC, or SKU.

This is useful for competitive price analysis, catalog deduplication, and supplier matching.

In [ ]:
# Simulate two store catalogs with overlapping products
store_a = [
    {"title": "Classic Percale Sheet Set - White",  "price": "149.00", "currency": "USD", "gtin": "0850031234567"},
    {"title": "Luxe Sateen Duvet Cover - Cream",    "price": "199.00", "currency": "USD", "gtin": "0850031234890"},
    {"title": "Down Alternative Comforter",          "price": "249.00", "currency": "USD", "gtin": "0850031234111"},
    {"title": "Waffle Bath Towel Set",               "price": "79.00",  "currency": "USD"},
]

store_b = [
    {"title": "Classic Percale Sheet Set (White)",   "price": "139.00", "currency": "USD", "gtin": "0850031234567"},
    {"title": "Luxe Sateen Duvet Cover, Cream",      "price": "209.00", "currency": "USD"},
    {"title": "Organic Cotton Pillowcase Pair",      "price": "49.00",  "currency": "USD"},
    {"title": "Waffle Towel Bath Set",               "price": "69.00",  "currency": "USD"},
]

# Fuzzy match by title
matches = shopextract.fuzzy_match(store_a, store_b, threshold=0.7)

print(f"Found {len(matches)} matching products across stores\n")
print(f"{'Store A':40s}  {'Store B':40s}  {'Similarity':>10s}")
print("=" * 95)
for prod_a, prod_b, score in matches:
    print(f"{prod_a['title'][:40]:<40s}  {prod_b['title'][:40]:<40s}  {score:>9.1%}")

In [ ]:
# Exact GTIN lookup -- find a specific product by barcode
all_products = store_a + store_b
gtin_matches = shopextract.match_gtin("0850031234567", all_products)

print(f"Products matching GTIN 0850031234567: {len(gtin_matches)}\n")
for m in gtin_matches:
    print(f"  {m['title']:<45s}  ${m['price']:>7s}  GTIN: {m['gtin']}")

---

## 5. Marketplace Validation

Works everywhere -- no network required.

Before listing products on a marketplace, you need to ensure they meet that platform's requirements. Each marketplace has different mandatory fields, character limits, and format rules.

`validate()` checks your product data against the rules for **Google Shopping**, **idealo**, **Amazon**, or **eBay** and returns a detailed report with errors and warnings.

In [ ]:
# Validate products for Google Shopping
report = shopextract.validate(products_as_dicts, marketplace="google_shopping")

print(f"Marketplace:  {report.marketplace}")
print(f"Total:        {report.total}")
print(f"Valid:        {report.valid}")
print(f"Invalid:      {report.invalid}")
print(f"Warnings:     {report.warnings}")
print(f"Pass Rate:    {report.pass_rate:.0f}%")

if report.issues:
    print(f"\nIssues Found:")
    print("-" * 70)
    for issue in report.issues:
        icon = "ERROR" if issue.severity == "error" else "WARN "
        print(f"  [{icon}] {issue.product_title}: {issue.field} -- {issue.error}")

In [ ]:
# Validate for idealo -- stricter requirements
report_idealo = shopextract.validate(products_as_dicts, marketplace="idealo")

print(f"idealo Validation: {report_idealo.valid}/{report_idealo.total} valid ({report_idealo.pass_rate:.0f}% pass rate)\n")
for issue in report_idealo.issues[:15]:  # Show first 15
    icon = "ERROR" if issue.severity == "error" else "WARN "
    print(f"  [{icon}] {issue.product_title}: {issue.field} -- {issue.error}")
if len(report_idealo.issues) > 15:
    print(f"  ... and {len(report_idealo.issues) - 15} more issues")

In [ ]:
# Validate for Amazon -- GTIN and brand are mandatory
report_amazon = shopextract.validate(products_as_dicts, marketplace="amazon")

print(f"Amazon Validation: {report_amazon.valid}/{report_amazon.total} valid ({report_amazon.pass_rate:.0f}% pass rate)\n")
for issue in report_amazon.issues[:15]:
    icon = "ERROR" if issue.severity == "error" else "WARN "
    print(f"  [{icon}] {issue.product_title}: {issue.field} -- {issue.error}")
if len(report_amazon.issues) > 15:
    print(f"  ... and {len(report_amazon.issues) - 15} more issues")

---

## 6. Price Monitoring

Works everywhere -- no network required.

`shopextract` can track product prices over time using lightweight SQLite snapshots. The workflow is:

1. **Snapshot** a store to capture current prices
2. Take another snapshot later (days, hours, minutes)
3. Run `changes()` to detect price increases, decreases, new products, and removed products
4. Run `price_history()` to see a specific product's price trajectory

Below we simulate this with two manual snapshots to demonstrate the API.

In [ ]:
import json
import sqlite3
import tempfile
from datetime import datetime, UTC, timedelta

# Create a temporary snapshot database with two snapshots
tmp_db = tempfile.NamedTemporaryFile(suffix=".db", delete=False)
tmp_db_path = tmp_db.name
tmp_db.close()

conn = sqlite3.connect(tmp_db_path)
conn.execute("""
    CREATE TABLE IF NOT EXISTS snapshots (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        domain TEXT NOT NULL,
        products_json TEXT NOT NULL,
        created_at TEXT NOT NULL
    )
""")

# Snapshot 1: Original prices (one week ago)
snapshot_v1 = [
    {"title": "Classic Percale Sheet Set",    "price": "149.00", "currency": "USD"},
    {"title": "Luxe Sateen Duvet Cover",      "price": "199.00", "currency": "USD"},
    {"title": "Down Alternative Comforter",    "price": "249.00", "currency": "USD"},
    {"title": "Waffle Bath Robe",              "price": "98.00",  "currency": "USD"},
]

# Snapshot 2: Current prices -- some changed, one new, one removed
snapshot_v2 = [
    {"title": "Classic Percale Sheet Set",    "price": "129.00", "currency": "USD"},  # Price drop
    {"title": "Luxe Sateen Duvet Cover",      "price": "219.00", "currency": "USD"},  # Price increase
    {"title": "Down Alternative Comforter",    "price": "249.00", "currency": "USD"},  # Unchanged
    {"title": "Organic Cotton Towel Bundle",   "price": "89.00",  "currency": "USD"},  # New product
    # "Waffle Bath Robe" removed
]

ts_old = (datetime.now(UTC) - timedelta(days=7)).isoformat()
ts_new = datetime.now(UTC).isoformat()

conn.execute("INSERT INTO snapshots (domain, products_json, created_at) VALUES (?, ?, ?)",
             ("demo.store", json.dumps(snapshot_v1), ts_old))
conn.execute("INSERT INTO snapshots (domain, products_json, created_at) VALUES (?, ?, ?)",
             ("demo.store", json.dumps(snapshot_v2), ts_new))
conn.commit()
conn.close()

print(f"Created snapshot database")
print(f"  Snapshot 1: {len(snapshot_v1)} products ({ts_old[:10]})")
print(f"  Snapshot 2: {len(snapshot_v2)} products ({ts_new[:10]})")

In [ ]:
# Detect changes between the two snapshots
detected = shopextract.changes("demo.store", db_path=tmp_db_path)

print(f"Detected {len(detected)} changes\n")
for change in detected:
    if change.change_type.value == "price_change":
        direction = "dropped" if change.new_price < change.old_price else "increased"
        print(f"  PRICE {direction.upper():>9s}  {change.title}")
        print(f"                    ${change.old_price} -> ${change.new_price} {change.currency}\n")
    elif change.change_type.value == "new_product":
        print(f"  NEW PRODUCT       {change.title}")
        print(f"                    ${change.price} {change.currency}\n")
    elif change.change_type.value == "removed_product":
        print(f"  REMOVED           {change.title}")
        print(f"                    Last price: ${change.last_price} {change.currency}\n")

In [ ]:
# Price history for a specific product across all snapshots
history = shopextract.price_history("demo.store", "Classic Percale Sheet Set", db_path=tmp_db_path)

print("Price History: Classic Percale Sheet Set")
print("=" * 45)
for ts, price in history:
    print(f"  {ts.strftime('%Y-%m-%d %H:%M')}   ${price:.2f}")

# Clean up temp database
import os
os.unlink(tmp_db_path)

---

## 7. Export

Works everywhere -- no network required.

Product data can be exported in multiple formats for downstream consumption:

| Format | Function | Use Case |
|---|---|---|
| **CSV** | `to_csv()` | Spreadsheets, data pipelines |
| **JSON** | `to_json()` | APIs, web applications |
| **Google Shopping XML** | `to_feed(format="google_shopping")` | Google Merchant Center feed |
| **Pandas DataFrame** | `to_dataframe()` | In-memory analysis |
| **Parquet** | `to_parquet()` | Columnar storage, big data |

All export functions accept a list of product dicts.

In [ ]:
import tempfile

# Export to CSV
csv_path = tempfile.mktemp(suffix=".csv")
shopextract.to_csv(products_as_dicts, csv_path)

with open(csv_path) as f:
    lines = f.readlines()

print("CSV Export (first 5 lines):")
print("-" * 80)
for line in lines[:5]:
    print(line.rstrip()[:120])

In [ ]:
# Export to JSON
json_path = tempfile.mktemp(suffix=".json")
shopextract.to_json(products_as_dicts, json_path, indent=2)

with open(json_path) as f:
    content = f.read()

print("JSON Export (first product):")
print("-" * 50)
import json as _json
parsed = _json.loads(content)
print(_json.dumps(parsed[0], indent=2))

In [ ]:
# Export to Google Shopping XML feed
feed_path = tempfile.mktemp(suffix=".xml")
shopextract.to_feed(products_as_dicts, feed_path, format="google_shopping")

with open(feed_path) as f:
    lines = f.readlines()

print("Google Shopping XML Feed (first 20 lines):")
print("-" * 60)
for line in lines[:20]:
    print(line.rstrip())

In [ ]:
# Export to Pandas DataFrame for interactive analysis
df = shopextract.to_dataframe(products_as_dicts)

df[["title", "price", "currency", "vendor", "sku", "gtin"]]

---

## 8. Quality Scoring

Works everywhere -- no network required.

`QualityScorer` evaluates how complete and useful each product record is on a 0.0 - 1.0 scale.

The scoring breakdown:
- **0.40** -- Base score for having a title
- **+0.15** -- Has a price
- **+0.15** -- Has an image
- **+0.15** -- Has a description
- **+0.15** -- Has a SKU or external ID

A score of **1.0** means the product has all key fields populated. Products without a title score **0.0**.

In [ ]:
# Score individual products
scorer = shopextract.QualityScorer()

print(f"{'Product':<42s}  {'Score':>5s}  Fields Present")
print("=" * 80)
for p in products_as_dicts:
    score = scorer.score_product(p)
    fields = []
    if p.get("title"):       fields.append("title")
    if p.get("price"):       fields.append("price")
    if p.get("image_url"):   fields.append("image")
    if p.get("description"): fields.append("desc")
    if p.get("sku"):         fields.append("sku")
    label = p.get("title", "(empty)")[:42]
    print(f"  {label:<42s}  {score:>4.2f}  {', '.join(fields)}")

In [ ]:
# Batch quality score -- average across all products
batch_score = scorer.score_batch(products_as_dicts)
print(f"Batch Average Quality: {batch_score:.2f}")
print(f"Products Scored:       {len(products_as_dicts)}")

---

## 9. Duplicate Detection

Works everywhere -- no network required.

`find_duplicates()` identifies duplicate products within a catalog using one of three methods:

| Method | How it works |
|---|---|
| `"title"` | Fuzzy string matching on product titles (configurable threshold) |
| `"gtin"` | Exact match on GTIN/EAN/UPC barcodes |
| `"sku"` | Exact match on SKU values |

Returns a list of `(index_a, index_b, similarity)` tuples for each duplicate pair found.

In [ ]:
# Create a catalog with intentional duplicates
catalog_with_dupes = [
    {"title": "Classic Percale Sheet Set - White",  "price": "149.00", "gtin": "0850031234567", "sku": "PERC-001"},
    {"title": "Luxe Sateen Duvet Cover - Cream",    "price": "199.00", "gtin": "0850031234890", "sku": "SATN-001"},
    {"title": "Classic Percale Sheet Set (White)",   "price": "149.00", "gtin": "0850031234567", "sku": "PERC-001-V2"},
    {"title": "Down Alternative Comforter",          "price": "249.00", "gtin": "0850031234111", "sku": "DOWN-001"},
    {"title": "Luxe Sateen Duvet - Cream",           "price": "209.00", "gtin": "0850031234999", "sku": "SATN-002"},
]

# Find duplicates by fuzzy title matching
title_dupes = shopextract.find_duplicates(catalog_with_dupes, method="title", threshold=0.8)

print(f"Title-Based Duplicates (threshold=0.8): {len(title_dupes)} pairs\n")
for idx_a, idx_b, sim in title_dupes:
    print(f"  [{idx_a}] {catalog_with_dupes[idx_a]['title']}")
    print(f"  [{idx_b}] {catalog_with_dupes[idx_b]['title']}")
    print(f"       Similarity: {sim:.1%}\n")

In [ ]:
# Find duplicates by exact GTIN match
gtin_dupes = shopextract.find_duplicates(catalog_with_dupes, method="gtin")

print(f"GTIN-Based Duplicates: {len(gtin_dupes)} pairs\n")
for idx_a, idx_b, sim in gtin_dupes:
    gtin = catalog_with_dupes[idx_a]["gtin"]
    print(f"  [{idx_a}] {catalog_with_dupes[idx_a]['title']}")
    print(f"  [{idx_b}] {catalog_with_dupes[idx_b]['title']}")
    print(f"       GTIN: {gtin}  (exact match)\n")

---

## Summary

This notebook demonstrated the core capabilities of `shopextract`:

1. **Platform Detection** -- Automatic identification of Shopify, WooCommerce, Magento, BigCommerce, Shopware, and custom stores
2. **Product Extraction** -- Tiered strategy (API > crawl > CSS) that adapts to each store's capabilities
3. **Catalog Analysis** -- Price statistics, brand distributions, and completeness scoring
4. **Product Matching** -- Fuzzy title matching and exact GTIN lookup across catalogs
5. **Marketplace Validation** -- Pre-listing checks for Google Shopping, idealo, Amazon, and eBay
6. **Price Monitoring** -- Snapshot-based change detection and price history tracking
7. **Multi-Format Export** -- CSV, JSON, Google Shopping XML, Pandas DataFrames, and Parquet
8. **Quality Scoring** -- Per-product and batch-level data quality assessment
9. **Duplicate Detection** -- Title similarity, GTIN, and SKU-based deduplication

### More to explore

- **`shopextract.watch(url)`** -- Real-time async generator that yields changes as they happen
- **`shopextract.compare(query, stores)`** -- Cross-store price comparison for a specific product
- **`shopextract.from_feed(feed_url)`** -- Import from Google Shopping XML/CSV feeds directly
- **`shopextract.to_parquet(products, path)`** -- Columnar export for large-scale analytics

Install it: `pip install shopextract`

Source: [github.com/umerkhan95/shopextract](https://github.com/umerkhan95/shopextract)